In [ ]:
import os
from datetime import datetime

import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from torch.nn.utils import get_total_norm, clip_grad_norm_

from torch.utils.tensorboard import SummaryWriter
from torchmetrics.text.rouge import ROUGEScore
from transformers import GPT2LMHeadModel
from tqdm import tqdm

from utils import get_tokenizer, generate_input_prompt

In [ ]:
DEVICE = torch.device("cuda:0")

In [ ]:
tokenizer = get_tokenizer()

In [ ]:
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.to(DEVICE)

# Create sft dataset objects

In [ ]:
sft_dataset_dedup_train = pd.read_parquet("datasets/sft/train_dedup.parquet")
sft_dataset_dedup_val = pd.read_parquet("datasets/sft/val_dedup.parquet")
sft_dataset_dedup_test = pd.read_parquet("datasets/sft/test_dedup.parquet")

sft_dataset_dedup_train.shape[0], sft_dataset_dedup_val.shape[0], sft_dataset_dedup_test.shape[0]

In [ ]:
sft_dataset_dedup_train["prompt"] = sft_dataset_dedup_train["post"].apply(generate_input_prompt)
sft_dataset_dedup_val["prompt"] = sft_dataset_dedup_val["post"].apply(generate_input_prompt)
sft_dataset_dedup_test["prompt"] = sft_dataset_dedup_test["post"].apply(generate_input_prompt)

In [ ]:
# sft_dataset_dedup_train["prompt_length"] = sft_dataset_dedup_train["prompt"].apply(
#     lambda x: len(tokenizer.encode(x))
# )
# prompt_length_mean, prompt_length_std = sft_dataset_dedup_train["prompt_length"].mean(), sft_dataset_dedup_train["prompt_length"].std()
# print(f"Prompt length: {prompt_length_mean:.2f} ± {prompt_length_std: .2f}")

Prompt length: 322.03 ±  93.22

In [ ]:
sft_dataset_dedup_train["summary_length"] = sft_dataset_dedup_train["summary"].apply(
    lambda x: len(tokenizer.encode(x))
)

sft_dataset_dedup_val["summary_length"] = sft_dataset_dedup_val["summary"].apply(
    lambda x: len(tokenizer.encode(x))
)

In [ ]:
def batched_sampling(batch):
    input_tensors = tokenizer(
        [
            prompt + summary
            for prompt, summary in zip(batch["prompt"], batch["summary"])
        ],
        padding=True, truncation=True, return_tensors="pt",
        add_special_tokens=True, padding_side="left"
    )

    input_tensors["summary_offset"] = torch.tensor(batch["summary_length"])
    return input_tensors

In [ ]:
# sft_train_dataset = Dataset.from_pandas(sft_dataset_dedup_train, split="train")
# sft_train_dataset = sft_train_dataset.map(batched_sampling, batched=True)
# sft_train_dataset.set_format(type="torch", columns=['input_ids', 'attention_mask', "summary_offset"])
# sft_train_dataset.save_to_disk("datasets/sft/hf_dataset/train")

# sft_val_dataset = Dataset.from_pandas(sft_dataset_dedup_val, split="train")
# sft_val_dataset = sft_val_dataset.map(batched_sampling, batched=True)
# sft_val_dataset.set_format(type="torch", columns=['input_ids', 'attention_mask', "summary_offset"])
# sft_val_dataset.save_to_disk("datasets/sft/hf_dataset/val")

In [ ]:
sft_train_dataset = (
    Dataset
    .load_from_disk("datasets/sft/hf_dataset/train")
    .with_format(
        "torch",
        columns=['input_ids', 'attention_mask', "summary_offset"],
        device=DEVICE,
        dtype=torch.int32
    )
)

sft_val_dataset = (
    Dataset
    .load_from_disk(
        "datasets/sft/hf_dataset/val"
    ).with_format(
        "torch",
        columns=['input_ids', 'attention_mask', "summary_offset"],
        device=DEVICE,
        dtype=torch.int32
    )
)

# Test loss computation

In [ ]:
test_batch = sft_val_dataset[:2]
summary_offset = test_batch.pop('summary_offset')

for k, v in test_batch.items():
    print(k, v.size())
    test_batch[k] = v.to(DEVICE)

In [ ]:
test_batch["input_ids"][0, -summary_offset[0] - 1:]

In [ ]:
test_batch["input_ids"][1, -summary_offset[1] - 1:]

In [ ]:
with torch.no_grad():
    res = model(
        **test_batch
    )

In [ ]:
max_offset = torch.max(summary_offset)

In [ ]:
logits = res.logits[:, -max_offset-1:-1]
op = torch.nn.LogSoftmax(dim=-1)
logprobas = op(logits)

labels = test_batch["input_ids"][:, -max_offset:].unsqueeze(2)

In [ ]:
label_logprobas = torch.gather(input=logprobas, dim=2, index=labels).squeeze()

In [ ]:
labels_mask = torch.zeros_like(label_logprobas)

In [ ]:
for i, offset in enumerate(summary_offset):
    for j in range(offset):
        labels_mask[i, - 1 - j] = 1

In [ ]:
loss = - (label_logprobas * labels_mask).sum(dim=-1) / labels_mask.sum(dim=-1)

# Training loop

In [ ]:
logsoftmax = torch.nn.LogSoftmax(dim=-1)

def compute_loss(model, batch, do_not_need_grad=False):
    input_ids, attention_mask, summary_offset = batch
    max_offset = torch.max(summary_offset)

    with torch.autograd.grad_mode.inference_mode(mode=do_not_need_grad):
        res = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    logits = res.logits[:, -max_offset-1:-1]
    logprobas = logsoftmax(logits)
    labels = input_ids[:, -max_offset:].unsqueeze(2)
    label_logprobas = torch.gather(input=logprobas, dim=2, index=labels).squeeze()

    labels_mask = torch.zeros_like(label_logprobas)

    for i, offset in enumerate(summary_offset):
        for j in range(offset):
            labels_mask[i, - 1 - j] = 1

    loss_per_sequence = - (label_logprobas * labels_mask).sum(dim=-1) / labels_mask.sum(dim=-1)
    return loss_per_sequence.mean()

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0
EVAL_EVERY = 1
LR = 3e-4
WD = 0
CLIP_GRAD_VAL = float("inf")

In [ ]:
def collate_fn(batch):
    input_ids = torch.nn.utils.rnn.pad_sequence(
        [_["input_ids"] for _ in batch], batch_first=True, padding_value=tokenizer.pad_token_id, padding_side='left'
    )

    attention_mask = torch.nn.utils.rnn.pad_sequence(
        [_["attention_mask"] for _ in batch], batch_first=True, padding_value=0,  padding_side='left'
    )
    summary_offset = torch.tensor([_["summary_offset"] for _ in batch])
    return input_ids, attention_mask, summary_offset

In [ ]:
sft_train_dl = torch.utils.data.DataLoader(sft_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
sft_val_dl = torch.utils.data.DataLoader(sft_val_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
iteration_number = 0
training_step = 0
current_time = datetime.now().strftime("%b%d_%H-%M-%S")
model_name = f"sft_{current_time}_lr={LR}_weight_decay={WD}_clip_grad_val={CLIP_GRAD_VAL}"
writer = SummaryWriter(log_dir=os.path.join("logs", model_name))

while True:
    model.train()
    for train_batch in tqdm(sft_train_dl, desc=f"Training iteration {iteration_number}"):
        training_step += 1
        training_loss = compute_loss(model=model, batch=train_batch)
        writer.add_scalar(tag="loss/train", scalar_value=training_loss, global_step=training_step)

        training_loss.backward()

        if CLIP_GRAD_VAL < float('inf'):
            grad_norm_before_clipping = clip_grad_norm_(parameters=model.parameters(), error_if_nonfinite=True, max_norm=10)
            writer.add_scalar(tag="grad_norm/before_clipping", scalar_value=grad_norm_before_clipping, global_step=training_step)

        grad_norm_after_clipping = get_total_norm(
            tensors=(p.grad for p in model.parameters() if p.grad is not None), error_if_nonfinite=True
        )
        writer.add_scalar(tag="grad_norm/after_clipping", scalar_value=grad_norm_after_clipping, global_step=training_step)

        optimizer.step()
        optimizer.zero_grad()

    if iteration_number % EVAL_EVERY == 0:
        model.eval()
        validation_loss = 0
        num_validation_batches = 0
        for val_batch in tqdm(sft_val_dl):
            validation_loss += compute_loss(model=model, batch=val_batch, do_not_need_grad=True)
            num_validation_batches += 1

        writer.add_scalar(
            tag="loss/val",
            scalar_value=validation_loss / num_validation_batches,
            global_step=iteration_number
        )

    iteration_number += 1

# Debug

In [ ]:
sft_test_dataset = Dataset.from_pandas(sft_dataset_dedup_test, split="test")

In [ ]:
def collate_fn(batch):
    res = tokenizer(
        [_["prompt"] for _ in batch], padding=True, truncation=True, return_tensors="pt", add_special_tokens=False, padding_side="left"
    )
    return res["input_ids"].to(DEVICE), res["attention_mask"].to(DEVICE), [_["summary"] for _ in batch]

In [ ]:
BATCH_SIZE = 256
NUM_WORKERS = 0

sft_test_dl = torch.utils.data.DataLoader(sft_test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)

In [ ]:
model.eval()
for (input_ids, attention_masks, reference_summaries) in tqdm(sft_test_dl):
    break

In [ ]:
with torch.no_grad():
    generated_tokens = model.generate(
        input_ids=input_ids, do_sample=False,
        attention_mask=attention_masks,
        output_attentions=True, return_dict_in_generate=True,
        max_new_tokens=256
    )

In [ ]:
batch_size, input_seq_len = input_ids.size()
_, output_seq_len = generated_tokens.sequences.size()
print(f"input seq size {input_seq_len} tokens, output seq size: {output_seq_len}, of which {output_seq_len - input_seq_len} are new tokens")

In [ ]:
attention_scores = generated_tokens.attentions  # len(attention_scores) == max_new_tokens, len(attention_scores[0]) == num_transformer_blocks

In [ ]:
attention_scores[0][0].size()  # (batch_size, num_heads, sequence_length, sequence_length)

In [ ]:
attention_scores[-1][0].size()  # (batch_size, num_heads, sequence_length, sequence_length)

In [ ]:
attention_masks.sum(dim=-1)

In [ ]:
sns.heatmap(attention_scores[-1][0][0][0].cpu().numpy())

In [ ]:
# output_tokens = generated_tokens.sequences[:, input_seq_len:]
# for i in range(batch_size):
#     input_i, output_i = input_ids[i].tolist(), output_tokens[i].tolist()
#     print(
#         tokenizer.decode([_ for _ in input_i + output_i if _ != tokenizer.pad_token_id])
#     )
#     print("*" * 25 + "\n")

In [ ]:
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config

In [ ]:
vanilla_generations = open("logs/vanilla/test_generations", "w")

In [ ]:
rouge = ROUGEScore()

model.eval()
for (input_ids, attention_masks, reference_summaries) in tqdm(sft_test_dl):
    batch_size, input_seq_len = input_ids.size()
    with torch.no_grad():
        generated_tokens = model.generate(
            input_ids=input_ids, do_sample=False,
            attention_mask=attention_masks,
            output_attentions=True, return_dict_in_generate=True,
            max_new_tokens=256
        )
    output_token_ids = generated_tokens.sequences[:, input_seq_len:].tolist()
    output_tokens_wo_padding = [
        [_  for _ in output_seq if _ != tokenizer.pad_token_id]
        for output_seq in output_token_ids
    ]
    output_seqs = tokenizer.decode(output_tokens_wo_padding)
    rouge.update(preds=output_seqs, target=reference_summaries)

    for input_seqs, output_seqs in zip(tokenizer.decode(input_ids, skip_special_tokens=True), output_seqs):
        vanilla_generations.write(f"Input: {input_seqs}, output: {output_seqs}\n")

In [ ]:
writer = SummaryWriter(log_dir=os.path.join("logs/vanilla"))
for k, v in rouge.compute().items():
    writer.add_scalar(tag=f"rouge_test/{k}", scalar_value=v)

In [ ]:
rouge.compute()

In [ ]:
print(rouge)

In [ ]:
rouge.plot()

In [ ]:
model = GPT2LMHeadModel.from_pretrained(
    '/home/logsumexp/workspace/rl-summarizer/artifacts/sft/Apr25_19-14-25_lr=3e-05_weight_decay=0_clip_grad_val=4/epoch_20'
)
model.to(DEVICE)

In [ ]:
optimizer_state = torch.load(
    '/home/logsumexp/workspace/rl-summarizer/artifacts/sft/Apr25_19-14-25_lr=3e-05_weight_decay=0_clip_grad_val=4/epoch_20/optimizer_state',
    device="cuda:0"
)

In [ ]:
with open('/home/logsumexp/workspace/rl-summarizer/artifacts/sft/Apr25_19-14-25_lr=3e-05_weight_decay=0_clip_grad_val=4/epoch_20/optimizer_state', "rb") as f:
    o_state = f.read()

In [ ]:
type(o_state)